# Capstone 3 — Write Like Shakespeare

*Module 4 capstone — ML & NLP by Data Trainers LLC*

---

## Welcome to the Final Capstone

You've made it. Over the past four modules you've built:

- **Module 1**: Classical NLP — topic modelling, NER, TF-IDF + LogReg classifiers
- **Module 2**: Dense word embeddings — Word2Vec CBOW, Doc2Vec, sentence-transformers
- **Module 3**: Neural text classification — MLP with learned embeddings, BERT fine-tuning
- **Module 4**: Text generation — character-level LSTM (NB 11) and GPT-2 fine-tuning (NB 12)

Now you put it all together. **Your mission**: build a Shakespeare text generator.

---

## The STAR Frame

| | |
|---|---|
| **Situation** | A literature startup wants a "Shakespeare remixer" for their creative writing app. They need a model that, given a seed phrase, continues in Shakespeare's style. |
| **Task** | Choose one implementation path (LSTM char-level **or** GPT-2 fine-tune), train a model on Tiny Shakespeare, and demo it. |
| **Action** | Load the ~1.1 MB public-domain text. Pick your path. Implement the full pipeline: preprocess → model → train → evaluate → generate. |
| **Result** | A working generator that produces 200 tokens of convincing Shakespearean text from a seed prompt. Plus: a 60-second non-engineer pitch and a 1-page technical memo. |

---

## Learning Objectives

1. Independently build a generation pipeline without full scaffolding
2. Understand the trade-off between char-level LSTM (from scratch) and subword GPT-2 (fine-tuned)
3. Orchestrate every step: data → vocab/tokenizer → model → train → eval → generate
4. Communicate ML results to both technical and non-technical audiences
5. Measure generation quality quantitatively (bits-per-char or test perplexity)

---

## Prerequisites

- NB 11: Character-level LSTM/GRU text generation (rental descriptions)
- NB 12: GPT-2 fine-tuning with HuggingFace Trainer
- PyTorch Primer (foundations-genai/PytorchPrimer)

**Estimated time:** ~60 minutes on Colab GPU for either path.

---

## GPU Setup (Google Colab)

Before running: `Runtime → Change runtime type → Hardware accelerator → T4 GPU`

## Section 0: Environment Setup

In [ ]:
# Install required packages (run this first in Google Colab)
# If running locally with a venv, you may skip this cell
!pip install -q torch transformers datasets accelerate evaluate matplotlib numpy

In [ ]:
# Core imports — visualisation -> data -> model -> training order
import os
import math
import random
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")
print("\nEnvironment setup complete!")

## Section 1: What Are We Building?

### The Tiny Shakespeare Dataset

**Tiny Shakespeare** is ~1.1 MB of plain text compiled from Shakespeare's plays — Hamlet, Macbeth, Romeo & Juliet, and more. It contains about:

- **1,115,394 characters** total
- **~65 unique characters** (letters, punctuation, newlines, spaces)
- Roughly **40,000 lines** of dialogue

It's a classic benchmark for character-level language models — small enough to train in minutes, but complex enough to learn meaningful patterns.

### Two Paths

| | Path A — Char-Level GRU | Path B — Fine-tuned GPT-2 |
|---|---|---|
| **What you learn** | Build a generative model from scratch | Fine-tune a pretrained LLM |
| **Granularity** | Character-level (~65 tokens) | Subword BPE (~50K tokens) |
| **Model** | `CharRNN` (Embedding→GRU→Linear) | `gpt2` from HuggingFace |
| **Training time** | ~10 min on T4 GPU | ~5 min on T4 GPU |
| **Metric** | Bits-per-character (BPC) | Test perplexity |
| **Output quality** | Good style, occasional gibberish | More coherent, less "from scratch" |

### Which Should You Pick?

- Pick **Path A** if you want to deeply understand how autoregressive sequence generation works from first principles.
- Pick **Path B** if you want to practice the modern HuggingFace Trainer workflow and understand transfer learning for generation.

You only need to complete **one path** for this capstone. The solution notebook implements both.

In [ ]:
# ── Load Tiny Shakespeare ─────────────────────────────────────
# Direct download from Andrej Karpathy's char-rnn repo (public domain)
SHAKESPEARE_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

if not os.path.exists('shakespeare.txt'):
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(SHAKESPEARE_URL, 'shakespeare.txt')
    print("Done.")

with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    full_text = f.read()

print(f"Total characters: {len(full_text):,}")
print(f"Unique characters: {len(set(full_text))}")
print(f"\nFirst 400 characters:")
print(full_text[:400])

## Path Choice Checkpoint

Before continuing, decide which path you will implement. Fill in the cell below.

In [ ]:
# ── STUDENT CHOICE ────────────────────────────────────────────
# Edit the variables below, then run this cell.

MY_PATH = None  # YOUR CODE: set to 'A' for char-level GRU, or 'B' for GPT-2 fine-tune

MY_JUSTIFICATION = None  # YOUR CODE: 1-2 sentences explaining your choice

# ── Verification ─────────────────────────────────────────────
assert MY_PATH in ('A', 'B'), "MY_PATH must be 'A' or 'B'"
assert isinstance(MY_JUSTIFICATION, str) and len(MY_JUSTIFICATION) > 10, "Provide a real justification"
print(f"Path selected: {MY_PATH}")
print(f"Justification: {MY_JUSTIFICATION}")

---

## Section 2 — Path A: Character-Level GRU

*(Skip to Section 3 if you chose Path B)*

### How Char-Level Generation Works

The idea is elegantly simple:

1. **Tokenize** the text into individual characters.
2. **Build a vocabulary** mapping each unique char to an integer ID.
3. **Create sliding windows**: for each window of length `seq_len`, the *input* is chars `[i..i+seq_len-1]` and the *target* is the same window shifted right by 1: `[i+1..i+seq_len]`. At every position, the model predicts the next character.
4. **Train** a GRU that learns to predict the next char distribution.
5. **Generate** by sampling from the predicted distribution at each step (with a temperature parameter to control creativity).

```python
# Example: seq_len = 5, text = "ROMEO"
# x = [R, O, M, E]   →   y = [O, M, E, O]
#      ↑ input chars        ↑ targets (next char for each input)
```

### Model Architecture

```
Embedding(vocab_size=65, emb_dim=64)
    ↓
GRU(input=64, hidden=256, num_layers=2, dropout=0.2, batch_first=True)
    ↓ (hidden states at every step)
Linear(256, vocab_size=65)    ← logits over the char vocabulary
```

### Hyperparameters

In [ ]:
# ── Path A Hyperparameters ────────────────────────────────────
CHAR_SEQ_LEN    = 100    # length of each training window
CHAR_EMB_DIM    = 64     # char embedding dimension (tiny vocab → tiny embedding)
CHAR_HIDDEN     = 256    # GRU hidden size
CHAR_NUM_LAYERS = 2      # stacked GRU layers
CHAR_DROPOUT    = 0.2    # dropout between GRU layers
CHAR_BATCH_SIZE = 64     # batch size
CHAR_EPOCHS     = 10     # training epochs
CHAR_LR         = 1e-3   # Adam learning rate
CHAR_CLIP       = 1.0    # gradient clipping threshold
GEN_MAX_NEW     = 200    # characters to generate
GEN_TEMP        = 0.8    # default sampling temperature
SEED_PHRASE     = 'ROMEO:'

print("Path A hyperparameters set.")

### Lab A1 — Build the Character Vocabulary

Build two dictionaries:
- `stoi`: maps each unique character to an integer index (string → int)
- `itos`: the reverse (int → string)
- `vocab_size`: number of unique characters
- `all_ids`: the entire text encoded as a `torch.long` tensor

**Hint**: `sorted(set(full_text))` gives you all unique chars in sorted order.

In [ ]:
# Lab A1: Build char vocab and encode the full text

# 1. Get all unique characters
chars = None  # YOUR CODE

# 2. Build stoi and itos mappings
stoi = None  # YOUR CODE
itos = None  # YOUR CODE

vocab_size = None  # YOUR CODE

# 3. Encode the entire text as a tensor of int IDs
all_ids = None  # YOUR CODE: torch.tensor([stoi[c] for c in full_text], dtype=torch.long)

# ── Verification ─────────────────────────────────────────────
assert vocab_size is not None and vocab_size >= 60, f"Expected ~65 chars, got {vocab_size}"
assert all_ids is not None and len(all_ids) == len(full_text)
print(f"Vocabulary size: {vocab_size}")
print(f"Encoded text length: {len(all_ids):,} token IDs")
print(f"First 10 chars: {''.join(chars[:10])}")
print(f"First 10 IDs:   {all_ids[:10].tolist()}")

### Lab A2 — Build the CharDataset

Create a PyTorch `Dataset` that yields `(x, y)` pairs of sliding windows.

- `__len__`: number of valid windows = `len(ids) - seq_len`
- `__getitem__(i)`: return `(ids[i:i+seq_len], ids[i+1:i+seq_len+1])`

In [ ]:
# Lab A2: CharDataset — sliding windows of length CHAR_SEQ_LEN

class CharDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = ids
        self.seq_len = seq_len

    def __len__(self):
        return None  # YOUR CODE

    def __getitem__(self, i):
        # Return (input_window, target_window) — each of length self.seq_len
        x = None  # YOUR CODE
        y = None  # YOUR CODE
        return x, y

# ── Split 90/10 for train/val ─────────────────────────────────
split_idx = int(0.9 * len(all_ids))
train_ids = all_ids[:split_idx]
val_ids   = all_ids[split_idx:]

train_ds_a = CharDataset(train_ids, CHAR_SEQ_LEN)
val_ds_a   = CharDataset(val_ids,   CHAR_SEQ_LEN)

train_loader_a = DataLoader(train_ds_a, batch_size=CHAR_BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader_a   = DataLoader(val_ds_a,   batch_size=CHAR_BATCH_SIZE, shuffle=False, num_workers=0)

# ── Verification ─────────────────────────────────────────────
x_b, y_b = next(iter(train_loader_a))
print(f"Batch x shape: {x_b.shape}")
print(f"Batch y shape: {y_b.shape}")
# Decode first example to sanity-check alignment
print(f"x[:20]: {''.join(itos[i] for i in x_b[0, :20].tolist())}")
print(f"y[:20]: {''.join(itos[i] for i in y_b[0, :20].tolist())}")

### Lab A3 — Define the CharRNN Model

Build `CharRNN(nn.Module)` with this architecture:

```
Embedding(vocab_size, emb_dim)
GRU(emb_dim, hidden, num_layers, dropout, batch_first=True)
Linear(hidden, vocab_size)
```

The `forward(x, h=None)` method should:
1. Embed `x` → shape `(B, T, emb_dim)`
2. Pass through GRU → `(out, h_new)`, `out` shape `(B, T, hidden)`
3. Project each time-step output to logits → shape `(B, T, vocab_size)`
4. Return `(logits, h_new)` — the new hidden state is needed for generation

In [ ]:
# Lab A3: CharRNN model

class CharRNN(nn.Module):
    def __init__(self, vocab_size, emb=CHAR_EMB_DIM, hidden=CHAR_HIDDEN,
                 layers=CHAR_NUM_LAYERS, dropout=CHAR_DROPOUT):
        super().__init__()
        self.emb = None  # YOUR CODE: nn.Embedding(...)
        self.gru = None  # YOUR CODE: nn.GRU(..., batch_first=True)
        self.fc  = None  # YOUR CODE: nn.Linear(...)

    def forward(self, x, h=None):
        # x: (B, T) integer token ids
        e   = None  # YOUR CODE: embed x
        out, h_new = None, None  # YOUR CODE: pass e through gru
        logits = None  # YOUR CODE: project to vocab
        return logits, h_new

# ── Instantiate and verify ────────────────────────────────────
model_a = CharRNN(vocab_size).to(device)
print(model_a)
n_params = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

# Quick forward pass check
x_test = torch.randint(0, vocab_size, (4, CHAR_SEQ_LEN)).to(device)
logits_test, h_test = model_a(x_test)
print(f"Output logits shape: {logits_test.shape}   (expect: [4, {CHAR_SEQ_LEN}, {vocab_size}])")
print(f"Hidden state shape:  {h_test.shape}        (expect: [{CHAR_NUM_LAYERS}, 4, {CHAR_HIDDEN}])")

### Lab A4 — Training Loop

Write a full training loop for `CHAR_EPOCHS` epochs:

1. Use `nn.CrossEntropyLoss()`
2. Use `torch.optim.Adam(lr=CHAR_LR)`
3. For each batch: zero grad → forward → reshape logits to `(B*T, V)` and targets to `(B*T,)` → loss → backward → **clip gradients** to `CHAR_CLIP` → step
4. After each epoch: compute validation loss and perplexity (`exp(val_loss)`)
5. Track and print: epoch, train loss, val loss, val perplexity

**Note on CrossEntropyLoss input shapes**:
- `logits` is `(B, T, V)` but `F.cross_entropy` expects `(N, C)` where N=B*T and C=V
- Reshape: `logits.view(-1, vocab_size)` and `targets.view(-1)`

In [ ]:
# Lab A4: Training loop for CharRNN

criterion_a = nn.CrossEntropyLoss()
optimizer_a = torch.optim.Adam(model_a.parameters(), lr=CHAR_LR)

history_a = {'train_loss': [], 'val_loss': [], 'val_ppl': []}

for epoch in range(1, CHAR_EPOCHS + 1):
    # ── Training phase ────────────────────────────────────────
    model_a.train()
    total_loss, n_batches = 0.0, 0
    for x_batch, y_batch in train_loader_a:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer_a.zero_grad()

        logits, _ = None  # YOUR CODE: forward pass

        # Reshape for cross entropy: (B*T, V) and (B*T,)
        loss = None  # YOUR CODE
        loss.backward()

        # Clip gradients to prevent exploding gradients
        None  # YOUR CODE: torch.nn.utils.clip_grad_norm_(...)

        optimizer_a.step()
        total_loss += loss.item()
        n_batches  += 1

    train_loss = total_loss / n_batches

    # ── Validation phase ──────────────────────────────────────
    model_a.eval()
    val_loss_total, val_batches = 0.0, 0
    with torch.no_grad():
        for x_v, y_v in val_loader_a:
            x_v, y_v = x_v.to(device), y_v.to(device)
            logits_v, _ = None  # YOUR CODE
            v_loss = None  # YOUR CODE: cross entropy, same reshaping
            val_loss_total += v_loss.item()
            val_batches    += 1

    val_loss = val_loss_total / val_batches
    val_ppl  = math.exp(val_loss)   # perplexity

    history_a['train_loss'].append(train_loss)
    history_a['val_loss'].append(val_loss)
    history_a['val_ppl'].append(val_ppl)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f}")

print("\nTraining complete!")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history_a['train_loss'], marker='o', label='train')
axes[0].plot(history_a['val_loss'],   marker='s', label='val')
axes[0].set_title('Loss curves (Path A)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].legend()

axes[1].plot(history_a['val_ppl'], marker='o', color='#FF7F50')
axes[1].set_title('Validation perplexity (Path A)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')

plt.tight_layout()
plt.show()

### Lab A5 — Generate Text with Temperature Sampling

Implement the `generate_char` function:

1. Encode the seed phrase using `stoi`
2. Feed it through the model to prime the hidden state
3. For each of `max_new` steps:
   - Take the last predicted logits
   - Apply temperature: divide logits by `temp`
   - Softmax → probability distribution
   - Sample one character index with `torch.multinomial`
   - Append it to the output
4. Decode the generated IDs back to a string

**Temperature intuition**: `temp < 1.0` → more conservative/repetitive; `temp > 1.0` → more creative/chaotic

In [ ]:
# Lab A5: Text generation with temperature

@torch.no_grad()
def generate_char(model, seed, max_new=GEN_MAX_NEW, temp=GEN_TEMP):
    """
    Generate `max_new` characters starting from `seed` string.
    Returns the seed + generated text as a single string.
    """
    model.eval()

    # 1. Encode seed phrase
    seed_ids = None  # YOUR CODE: [stoi[c] for c in seed]
    input_tensor = torch.tensor(seed_ids, dtype=torch.long).unsqueeze(0).to(device)  # (1, L)

    # 2. Prime hidden state by running seed through model
    logits, h = None  # YOUR CODE: forward on input_tensor

    generated = list(seed)

    # Use the last logit as the first prediction
    last_logit = logits[0, -1, :]   # shape (vocab_size,)

    for _ in range(max_new):
        # 3a. Apply temperature
        scaled = None  # YOUR CODE: last_logit / temp

        # 3b. Softmax to probabilities
        probs = None  # YOUR CODE

        # 3c. Sample one token
        next_id = None  # YOUR CODE: torch.multinomial(probs, num_samples=1)

        # 3d. Decode and append
        generated.append(itos[next_id.item()])

        # 3e. Feed the new token back in (shape: (1, 1))
        x_next = next_id.unsqueeze(0)              # (1, 1)
        logits, h = None  # YOUR CODE: forward on x_next, passing h
        last_logit = logits[0, -1, :]

    return ''.join(generated)

# ── Generate and print ────────────────────────────────────────
if model_a is not None:
    generated_text = generate_char(model_a, SEED_PHRASE, max_new=GEN_MAX_NEW, temp=GEN_TEMP)
    print("=" * 60)
    print(f"Seed: '{SEED_PHRASE}'")
    print("=" * 60)
    print(generated_text)
    print("=" * 60)

In [ ]:
# Demo: Temperature sweep — see how temperature controls creativity vs coherence
# (This cell is provided — just run it)

for temp in [0.3, 0.7, 1.0, 1.5]:
    sample = generate_char(model_a, 'ROMEO:', max_new=80, temp=temp)
    print(f"\n--- Temperature = {temp} ---")
    print(sample)

### Lab A6 — Bits-Per-Character

Bits-per-character (BPC) is the char-level analog of perplexity:

$$\text{BPC} = \frac{\text{avg cross-entropy loss}}{\ln 2}$$

Compute BPC on the held-out validation split. A well-trained char-level LSTM on Shakespeare typically reaches ~1.3–1.5 BPC.

*(Reminder: `math.log(2)` = `ln(2)` ≈ 0.693)*

In [ ]:
# Lab A6: Bits-per-character on validation set

def compute_bpc(model, loader):
    """Compute bits-per-character = avg_cross_entropy / ln(2)."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    with torch.no_grad():
        for x_b, y_b in loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            logits, _ = None  # YOUR CODE
            loss = None  # YOUR CODE: cross entropy, reshaped
            total_loss += loss.item()
            n_batches  += 1

    avg_loss = total_loss / n_batches
    bpc = avg_loss / math.log(2)
    return bpc

bpc_val = compute_bpc(model_a, val_loader_a)
print(f"Validation BPC: {bpc_val:.4f}")
print(f"(A well-trained LSTM typically reaches ~1.3–1.5 BPC on Shakespeare)")

In [ ]:
# Save the Path A model
torch.save(model_a.state_dict(), 'shakespeare_char_rnn.pt')
print("Model saved to shakespeare_char_rnn.pt")

---

## Section 3 — Path B: Fine-tune GPT-2

*(Skip to Section 4 if you chose Path A)*

### Why Fine-tuning GPT-2?

GPT-2 was pretrained on ~40 GB of internet text using the same next-token prediction objective. On 1.1 MB of Shakespeare, training from scratch would require far more data — but we can **fine-tune** the pretrained weights to shift its "style" toward Shakespeare.

The approach:
1. Load the `gpt2` tokenizer and model (117M params, smallest variant)
2. Tokenize the Shakespeare text into subword BPE tokens
3. Split into fixed-length chunks of `MAX_LENGTH=128` tokens
4. Fine-tune with HuggingFace `Trainer` for 3 epochs
5. Generate text with `model.generate()` using top-p sampling

### Key Difference from Path A

| | Path A | Path B |
|---|---|---|
| Starting point | Random weights | Pretrained GPT-2 weights |
| Token type | Characters (~65) | Subwords (~50,257) |
| Training data needed | More (learning from scratch) | Less (just shifting style) |
| Control | Full — you wrote it | Less — `Trainer` handles most |

### Hyperparameters

In [ ]:
# ── Path B Hyperparameters ────────────────────────────────────
MODEL_NAME    = 'gpt2'     # smallest GPT-2 (117M params)
MAX_LENGTH    = 128        # token chunk size for training
NUM_EPOCHS_B  = 3          # fine-tuning epochs
BATCH_SIZE_B  = 4          # per-device batch size (keep small for Colab)
GRAD_ACCUM    = 4          # accumulate gradients across batches (effective BS=16)
LR_B          = 5e-5       # learning rate
GEN_TEMP_B    = 0.8        # generation temperature
GEN_TOP_P     = 0.9        # top-p (nucleus) sampling
GEN_MAX_NEW_B = 200        # tokens to generate

print("Path B hyperparameters set.")

In [ ]:
# Import HuggingFace libraries for Path B
from transformers import (
    GPT2LMHeadModel, GPT2TokenizerFast,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset as HFDataset

# Load the GPT-2 tokenizer
tokenizer_b = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
# GPT-2 has no pad token by default — use eos_token
tokenizer_b.pad_token = tokenizer_b.eos_token

print(f"Tokenizer vocab size: {tokenizer_b.vocab_size:,}")
print(f"EOS token: '{tokenizer_b.eos_token}' (id={tokenizer_b.eos_token_id})")

# Quick demo: tokenize one line
sample_line = "To be, or not to be, that is the question."
tokens = tokenizer_b(sample_line)
print(f"\nSample: '{sample_line}'")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Decoded back: {tokenizer_b.decode(tokens['input_ids'])}")

### Lab B1 — Tokenize Shakespeare and Build Training Chunks

1. Tokenize the full Shakespeare text with `tokenizer_b(full_text, return_tensors='pt')`
2. Split the resulting `input_ids` into non-overlapping chunks of `MAX_LENGTH` tokens
3. Build a HuggingFace `Dataset` from the list of chunks
4. Split 90/10 into train/val

**Why chunks?** The HuggingFace Trainer expects a batch of fixed-length token sequences. We slide a non-overlapping window across the tokenized text.

In [ ]:
# Lab B1: Tokenize and chunk Shakespeare for GPT-2 training

# 1. Tokenize the full text (no truncation — we want all tokens)
all_token_ids = None  # YOUR CODE: tokenizer_b(full_text, return_tensors='pt')['input_ids'][0]
print(f"Total tokens in Shakespeare: {len(all_token_ids):,}")

# 2. Split into non-overlapping chunks of MAX_LENGTH tokens
def chunk_tokens(token_ids, chunk_size):
    """Split token_ids tensor into chunks of chunk_size. Drop the last partial chunk."""
    chunks = []
    for i in range(0, len(token_ids) - chunk_size, chunk_size):
        chunk = None  # YOUR CODE: token_ids[i : i + chunk_size].tolist()
        chunks.append({'input_ids': chunk})
    return chunks

all_chunks = chunk_tokens(all_token_ids, MAX_LENGTH)
print(f"Total training chunks: {len(all_chunks):,}")

# 3. Build HuggingFace Dataset and split 90/10
split_n = int(0.9 * len(all_chunks))
train_chunks = all_chunks[:split_n]
val_chunks   = all_chunks[split_n:]

train_hf = HFDataset.from_list(train_chunks)
val_hf   = HFDataset.from_list(val_chunks)

print(f"Train chunks: {len(train_hf):,}   Val chunks: {len(val_hf):,}")

# ── Verification ─────────────────────────────────────────────
sample_chunk = train_hf[0]['input_ids']
print(f"\nFirst chunk decoded:")
print(tokenizer_b.decode(sample_chunk[:50]))

### Lab B2 — Fine-tune with HuggingFace Trainer

Use `Trainer` to fine-tune GPT-2 on the Shakespeare chunks.

Key steps:
1. Load `GPT2LMHeadModel.from_pretrained(MODEL_NAME)`
2. Set up `TrainingArguments` (output_dir, epochs, batch_size, gradient_accumulation_steps, LR, eval_strategy)
3. Create a `DataCollatorForLanguageModeling(tokenizer_b, mlm=False)` — GPT-2 uses causal (not masked) LM
4. Create `Trainer` and call `.train()`

In [ ]:
# Lab B2: Fine-tune GPT-2 with HuggingFace Trainer

# 1. Load the pretrained GPT-2 model
model_b = None  # YOUR CODE: GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model_b.resize_token_embeddings(len(tokenizer_b))  # in case pad token changes vocab size

# 2. Training arguments
training_args = TrainingArguments(
    output_dir              = './shakespeare_gpt2',
    num_train_epochs        = NUM_EPOCHS_B,
    per_device_train_batch_size = BATCH_SIZE_B,
    per_device_eval_batch_size  = BATCH_SIZE_B,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate           = LR_B,
    eval_strategy           = 'epoch',    # evaluate after each epoch
    save_strategy           = 'no',       # don't save checkpoints (saves space)
    logging_steps           = 50,
    fp16                    = torch.cuda.is_available(),  # use mixed precision on GPU
    seed                    = SEED,
    report_to               = 'none',     # disable wandb/tensorboard
)

# 3. Data collator for causal language modelling
data_collator_b = DataCollatorForLanguageModeling(
    tokenizer = tokenizer_b,
    mlm       = False,   # False = causal LM (GPT-2 style)
)

# 4. Create and run Trainer
trainer_b = Trainer(
    model           = model_b,
    args            = training_args,
    train_dataset   = train_hf,
    eval_dataset    = val_hf,
    data_collator   = data_collator_b,
    tokenizer       = tokenizer_b,
)

print(f"Model parameters: {sum(p.numel() for p in model_b.parameters()):,}")
print("Starting fine-tuning...")
trainer_b.train()

### Lab B3 — Generate from Fine-tuned GPT-2

Use `model.generate()` with top-p (nucleus) sampling to generate Shakespeare text.

Parameters to use:
- `temperature=GEN_TEMP_B`
- `top_p=GEN_TOP_P`
- `do_sample=True`
- `max_new_tokens=GEN_MAX_NEW_B`

In [ ]:
# Lab B3: Generate text from fine-tuned GPT-2

def generate_gpt2(model, tokenizer, seed, max_new=GEN_MAX_NEW_B, temp=GEN_TEMP_B, top_p=GEN_TOP_P):
    """Generate text from fine-tuned GPT-2 using top-p sampling."""
    model.eval()
    gen_device = next(model.parameters()).device

    # Encode the seed prompt
    inputs = None  # YOUR CODE: tokenizer(seed, return_tensors='pt').to(gen_device)

    # Generate
    with torch.no_grad():
        output_ids = None  # YOUR CODE: model.generate(
                                         # inputs['input_ids'],
                                         # max_new_tokens=max_new,
                                         # temperature=temp,
                                         # top_p=top_p,
                                         # do_sample=True,
                                         # pad_token_id=tokenizer.eos_token_id
                                         # )

    # Decode and return
    return None  # YOUR CODE: tokenizer.decode(output_ids[0], skip_special_tokens=True)

# ── Generate ──────────────────────────────────────────────────
model_b.to(device)
text_finetuned = generate_gpt2(model_b, tokenizer_b, 'ROMEO:', max_new=GEN_MAX_NEW_B)
print("=" * 60)
print("Fine-tuned GPT-2 output:")
print("=" * 60)
print(text_finetuned)

### Lab B4 — Compare Fine-tuned vs Base GPT-2

Load the **base (un-fine-tuned)** GPT-2 and generate from the same seed. Does fine-tuning visibly shift the style toward Shakespeare?

In [ ]:
# Lab B4: Compare fine-tuned vs base GPT-2

# Load fresh base GPT-2 (no Shakespeare fine-tuning)
base_gpt2 = None  # YOUR CODE: GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

text_base = generate_gpt2(base_gpt2, tokenizer_b, 'ROMEO:', max_new=GEN_MAX_NEW_B)

print("=" * 60)
print("BASE GPT-2 (not fine-tuned):")
print("=" * 60)
print(text_base)
print()
print("=" * 60)
print("FINE-TUNED GPT-2 (Shakespeare):")
print("=" * 60)
print(text_finetuned)

### Lab B5 — Compute Test Perplexity

Use the `Trainer.evaluate()` method on the validation split to get the eval loss, then compute perplexity:

$$\text{Perplexity} = e^{\text{eval\_loss}}$$

In [ ]:
# Lab B5: Evaluate test perplexity

eval_results = trainer_b.evaluate()
eval_loss = eval_results['eval_loss']
perplexity_b = None  # YOUR CODE: math.exp(eval_loss)

print(f"Eval loss:    {eval_loss:.4f}")
print(f"Perplexity:   {perplexity_b:.2f}")

# Save the fine-tuned model
trainer_b.save_model('./shakespeare_gpt2_final')
print("\nFine-tuned model saved to ./shakespeare_gpt2_final")

---

## Section 4: Present Your Results

This is where ML meets communication. Two deliverables:

1. **Technical memo** — for your engineering team. Numbers, trade-offs, decisions.
2. **Non-technical pitch** — for the literature startup's product team. 3 sentences max.

### Lab P1 — Write the Technical Memo

In [ ]:
# Lab P1: Technical memo
# Fill in the variables below, then run to display.

PATH_CHOSEN   = None  # YOUR CODE: 'A' or 'B'
WHY_CHOSEN    = None  # YOUR CODE: 1-2 sentences on why you picked this path

# Fill in your actual numbers from training:
METRIC_NAME   = None  # YOUR CODE: 'Bits-per-character' (Path A) or 'Perplexity' (Path B)
METRIC_VALUE  = None  # YOUR CODE: your actual score

TRAINING_TIME = None  # YOUR CODE: approximate training time in minutes
MODEL_SIZE_MB = None  # YOUR CODE: model file size in MB (check the saved file)

TRADEOFFS     = None  # YOUR CODE: 2-3 sentences on trade-offs vs the other path

# ── Verification & display ────────────────────────────────────
assert all(x is not None for x in [PATH_CHOSEN, WHY_CHOSEN, METRIC_NAME, METRIC_VALUE,
                                    TRAINING_TIME, MODEL_SIZE_MB, TRADEOFFS])

memo = f"""
=== TECHNICAL MEMO: Shakespeare Generator ===

Path implemented: {PATH_CHOSEN}
Reason for choice: {WHY_CHOSEN}

Evaluation metric:
  {METRIC_NAME}: {METRIC_VALUE}

Resource usage:
  Training time: ~{TRAINING_TIME} minutes
  Model size:    ~{MODEL_SIZE_MB} MB

Trade-offs vs alternative path:
  {TRADEOFFS}
"""
print(memo)

### Lab P2 — Write the Non-Technical Pitch

Write a 3-sentence pitch for the literature startup's product team. They are not engineers.

**Guide**:
- Sentence 1: What the model learned (at a high level, no jargon)
- Sentence 2: How it works in practice (given a start, it continues the story)
- Sentence 3: Show them — paste one impressive generated sample

In [ ]:
# Lab P2: Non-technical pitch

PITCH_SENTENCE_1 = None  # YOUR CODE: What did the model learn?
PITCH_SENTENCE_2 = None  # YOUR CODE: How does it work in practice?
PITCH_SENTENCE_3 = None  # YOUR CODE: A generated example (copy from above)

# ── Verification & display ────────────────────────────────────
assert all(x is not None for x in [PITCH_SENTENCE_1, PITCH_SENTENCE_2, PITCH_SENTENCE_3])

print("=== 60-SECOND PITCH ===")
print()
print(PITCH_SENTENCE_1)
print(PITCH_SENTENCE_2)
print()
print("Here's what it produces:")
print(f'  "{PITCH_SENTENCE_3[:200]}"')

---

## Self-Check Quiz

Answer these before submitting. No code needed — just read and think.

1. **You have 1.1 MB of text. Would you rather train a char-level LSTM from scratch or fine-tune GPT-2? Defend your answer.**
   *(Hint: think about what each approach needs and already has)*

2. **In Path A, what is the expected initial cross-entropy loss at random initialization?**
   *(Hint: at random init, the model assigns equal probability to all `vocab_size` characters)*

3. **Your Path A output is `"aaabbbcccddd..."`  (highly repetitive). Name two likely bugs.**

4. **Your Path B output is indistinguishable from base GPT-2's output. What does this tell you about your fine-tune?**

---

## Congratulations!

You have completed the full ML & NLP course by Data Trainers LLC.

Here is everything you can now do:

| Module | Skills |
|--------|--------|
| 1 — Pre-NLP | Topic modelling (LDA), NER with spaCy, TF-IDF, logistic regression, boosting |
| 2 — Text Similarity | Word2Vec CBOW from scratch, Doc2Vec, sentence-transformers fine-tuning |
| 3 — Text Classification | MLP with learned embeddings, BERT fine-tuning with HuggingFace Trainer |
| 4 — Text Generation | Char-level GRU autoregressive models, GPT-2 fine-tuning, temperature sampling |

### Where to Go Next

- **foundations-genai** — PyTorch tensor ops, autograd, GPU, DataLoader deep dives
- **chatbots-with-genai** — LangChain, LangGraph, RAG, conversational agents
- **genai_and_llms** — LoRA, PEFT, quantization, prompt engineering at scale

You've earned it. Now go build something.